In [ ]:

import os
import pickle
import logging
from datetime import datetime

from database_training.database_process_single_file import process_single_file

In [ ]:
import os
import pickle
import logging
from datetime import datetime

from database_training.database_process_single_file import process_single_file

# =====================================================
# CONFIGURATION
# =====================================================
SCAN_ROOT = r"E:\\"                    # 🔴 Change if needed
OUTPUT_FOLDER = r"output_csv_files"
PICKLE_DB_PATH = r"database_backup/email_patterns_version3.pkl"
LOG_DIR = "email_db_logs"
SCANNED_FILES_TXT = r"scanned_files.txt"   # 🔴 UPDATE THIS

SUPPORTED_EXTENSIONS = (".csv", ".xls", ".xlsx")

# =====================================================
# SETUP DIRECTORIES
# =====================================================
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(PICKLE_DB_PATH), exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =====================================================
# LOGGING SETUP
# =====================================================
log_file = os.path.join(
    LOG_DIR, f"email_db_scan_{datetime.now().strftime('%Y-%m-%d')}.log"
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("EmailDBScanner")

# =====================================================
# INITIALIZE PICKLE DB IF MISSING
# =====================================================
if not os.path.exists(PICKLE_DB_PATH):
    with open(PICKLE_DB_PATH, "wb") as f:
        pickle.dump({}, f)
    logger.info("📦 Created new pickle database")

# =====================================================
# UTILITY: GET CURRENT DB COUNT
# =====================================================
def get_db_count():
    try:
        with open(PICKLE_DB_PATH, "rb") as f:
            return len(pickle.load(f))
    except Exception:
        return 0

# =====================================================
# UTILITY: LOAD ALREADY SCANNED FILES
# =====================================================
def load_scanned_files():
    if not os.path.exists(SCANNED_FILES_TXT):
        logger.warning("⚠️ scanned_files.txt not found — no files will be skipped")
        return set()

    with open(SCANNED_FILES_TXT, "r", encoding="utf-8", errors="ignore") as f:
        scanned_files = {
            os.path.normpath(line.strip())
            for line in f
            if line.strip()
        }

    logger.info(f"⏭️ Loaded {len(scanned_files)} already scanned files")
    return scanned_files

# =====================================================
# MAIN SCAN LOGIC
# =====================================================
def scan_and_build_db():
    total_files = 0
    processed_files = 0
    skipped_files = 0
    already_done_files = 0

    SKIP_FOLDERS = {
        "$RECYCLE.BIN",
        "System Volume Information",
        "Windows",
        "$WinREAgent"
    }

    # ⏭️ Load skip list
    scanned_files = load_scanned_files()

    logger.info(f"🚀 Starting scan on drive: {SCAN_ROOT}")

    for root, dirs, files in os.walk(SCAN_ROOT):

        # 🔒 Skip system folders
        dirs[:] = [
            d for d in dirs
            if d not in SKIP_FOLDERS and not d.startswith("$")
        ]

        for file in files:
            total_files += 1
            file_path = os.path.normpath(os.path.join(root, file))

            # Skip unsupported file types
            if not file.lower().endswith(SUPPORTED_EXTENSIONS):
                continue

            # ⏭️ Skip already processed files
            if file_path in scanned_files:
                already_done_files += 1
                logger.info(f"⏭️ Already processed, skipping: {file_path}")
                continue

            logger.info(f"📄 Found file: {file_path}")

            try:
                process_single_file(
                    input_file=file_path,
                    output_folder=OUTPUT_FOLDER,
                    output_pkl_path=PICKLE_DB_PATH
                )

                processed_files += 1
                db_count = get_db_count()

                logger.info(f"✅ Added patterns | File: {file} | Folder: {root}")
                logger.info(f"📊 Current database size: {db_count} companies")

            except Exception as e:
                skipped_files += 1
                logger.error(
                    f"❌ Skipped file | File: {file} | Reason: {str(e)}"
                )

    # =====================================================
    # FINAL SUMMARY
    # =====================================================
    logger.info("==============================================")
    logger.info("🏁 SCAN COMPLETED")
    logger.info(f"📁 Total files scanned        : {total_files}")
    logger.info(f"✅ Successfully processed     : {processed_files}")
    logger.info(f"⏭️ Already done (skipped)     : {already_done_files}")
    logger.info(f"⚠️ Failed / error files       : {skipped_files}")
    logger.info(f"📦 Final DB count             : {get_db_count()}")
    logger.info("==============================================")

# =====================================================
# RUN
# =====================================================
if __name__ == "__main__":
    scan_and_build_db()


In [1]:
import os
import pandas as pd

# =====================================================
# CONFIG
# =====================================================
SCAN_ROOT = r"E:\\"
OUTPUT_SUMMARY_CSV = "file_company_summary.csv"

SUPPORTED_EXTENSIONS = (".csv", ".xls", ".xlsx")
REQUIRED_COLUMNS = {"Company", "Contact Name", "Email"}

SKIP_FOLDERS = {
    "$RECYCLE.BIN",
    "System Volume Information",
    "Windows",
    "$WinREAgent"
}

LOG_EVERY = 200

summary_rows = []

# =====================================================
# COUNTERS
# =====================================================
total_files_seen = 0
candidate_files = 0
valid_files = 0
skipped_missing_columns = 0
failed_files = 0

# =====================================================
# SCAN FILES
# =====================================================
print("🚀 Scan started...")




🚀 Scan started...


In [3]:
for root, dirs, files in os.walk(SCAN_ROOT):

    dirs[:] = [
        d for d in dirs
        if d not in SKIP_FOLDERS and not d.startswith("$")
    ]

    for file in files:
        total_files_seen += 1
        print (total_files_seen)

        if not file.lower().endswith(SUPPORTED_EXTENSIONS):
            continue

        candidate_files += 1
        file_path = os.path.join(root, file)

        try:
            # Read file
            if file.lower().endswith(".csv"):
                df = pd.read_csv(
                    file_path,
                    usecols=["Company"],
                    low_memory=False
                )
            else:
                df = pd.read_excel(file_path)

            # Check required columns
            if not REQUIRED_COLUMNS.issubset(df.columns):
                skipped_missing_columns += 1
                continue

            total_rows = len(df)
            unique_companies = df["Company"].dropna().nunique()

            # ✅ PRINT IMMEDIATELY FOR VALID FILE
            print(
                f"\n✅ Valid file found:\n"
                f"   📄 File: {file_path}\n"
                f"   📊 Total rows: {total_rows}\n"
                f"   🏢 Unique companies: {unique_companies}"
            )

            summary_rows.append({
                "Folder Name": root,
                "File Name": file,
                "Total Rows": total_rows,
                "Unique Company Count": unique_companies
            })

            valid_files += 1

        except Exception as e:
            failed_files += 1
            print(f"❌ Failed to read: {file_path} | {e}")

        # 🔹 PERIODIC PROGRESS LOG
        if candidate_files % LOG_EVERY == 0:
            print(
                f"\n📊 Progress | "
                f"Seen: {total_files_seen} | "
                f"Candidates: {candidate_files} | "
                f"Valid: {valid_files} | "
                f"Skipped: {skipped_missing_columns} | "
                f"Failed: {failed_files}"
            )

# =====================================================
# SAVE SUMMARY CSV
# =====================================================
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)

# =====================================================
# FINAL SUMMARY
# =====================================================
print("\n🏁 Scan completed")
print(f"📁 Total files seen            : {total_files_seen}")
print(f"📄 Candidate CSV/Excel files   : {candidate_files}")
print(f"✅ Valid files processed       : {valid_files}")
print(f"⏭️ Skipped (missing columns)   : {skipped_missing_columns}")
print(f"❌ Failed to read              : {failed_files}")
print(f"💾 Summary saved to            : {OUTPUT_SUMMARY_CSV}")

255
256
257
258
❌ Failed to read: E:\\automatic_email_format\csv_filedd.csv | Usecols do not match columns, columns expected but not found: ['Company']
259
260
261
262
❌ Failed to read: E:\\automatic_email_format\GoogleColabfile_1.csv | Usecols do not match columns, columns expected but not found: ['Company']
263
❌ Failed to read: E:\\automatic_email_format\GoogleColabfile_2.csv | Usecols do not match columns, columns expected but not found: ['Company']
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
297
298
299
300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
394
395
396
397
398
39

KeyboardInterrupt: 